## _import libraries_

In [1]:
import pandas as pd
import numpy as np
import re
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## _load Dataset_

In [2]:
movies  = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
movies.head(10)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [3]:
ratings.head(10)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [4]:
print(movies.info())
print(ratings.info())

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None


## _cleaning Data_

In [5]:
print(movies.isnull().sum())
print(ratings.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [6]:
print(movies.duplicated().sum())
print(ratings.duplicated().sum())


0
0


In [7]:
print(movies.isna().sum())
print(ratings.isna().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [8]:
ratings.drop(columns=['timestamp'], inplace=True)

In [9]:
#  Clean Text (Genres)
def clean_genres(text):
    text = str(text).lower()
    text = text.replace('|', ' ')
    text = re.sub(r'[^a-z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

movies['genres'] = movies['genres'].apply(clean_genres)

In [10]:
#  Extract Year from Title
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    if match:
        return int(match.group(1))
    return np.nan

movies['year'] = movies['title'].apply(extract_year)

In [11]:
# Clean Title (Remove Year)

def clean_title(title):
    title = re.sub(r'\(\d{4}\)', '', str(title))  # remove year
    return title.strip().lower()

movies['clean_title'] = movies['title'].apply(clean_title)

In [12]:
 # Filter Rare Movies
movie_rating_count = ratings.groupby('movieId')['rating'].count()

popular_movies = movie_rating_count[movie_rating_count >= 20].index

ratings = ratings[ratings['movieId'].isin(popular_movies)]
movies = movies[movies['movieId'].isin(popular_movies)]

In [13]:
data = pd.merge(ratings, movies, on="movieId")
data.head(10)

,userId,movieId,rating,title,genres,year,clean_title
0,1,1,4.0,Toy Story (1995),adventure animation children comedy fantasy,1995.0,toy story
1,1,3,4.0,Grumpier Old Men (1995),comedy romance,1995.0,grumpier old men
2,1,6,4.0,Heat (1995),action crime thriller,1995.0,heat
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),mystery thriller,1995.0,seven (a.k.a. se7en)
4,1,50,5.0,"Usual Suspects, The (1995)",crime mystery thriller,1995.0,"usual suspects, the"
5,1,70,3.0,From Dusk Till Dawn (1996),action comedy horror thriller,1996.0,from dusk till dawn
6,1,101,5.0,Bottle Rocket (1996),adventure comedy crime romance,1996.0,bottle rocket
7,1,110,4.0,Braveheart (1995),action drama war,1995.0,braveheart
8,1,151,5.0,Rob Roy (1995),action drama romance war,1995.0,rob roy
9,1,163,5.0,Desperado (1995),action romance western,1995.0,desperado


## Collaborative Filtering

In [14]:
# prepare data for surprise library
reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']],reader)

In [15]:
# split data 80% train - 20% test
trainset, testset = train_test_split(surprise_data, test_size=0.2, random_state=42)

In [16]:
# train SVD model
model=SVD(n_factors=50,n_epochs=20, random_state=42)
model.fit(trainset)

In [17]:
# predict and evaluate model
predictions=model.test(testset)
actual    = [pred.r_ui for pred in predictions]
estimated = [pred.est  for pred in predictions]
rmse = root_mean_squared_error(actual, estimated)
mae  = mean_absolute_error(actual, estimated)
print("RMSE:",rmse)
print("MAE:",mae )

RMSE: 0.8394569373975963
MAE: 0.6423426926962722


In [18]:
def recommend_movies(user_id, n=10):
    # all movies
    all_movie_ids = ratings['movieId'].unique()
    # movies user already watched
    watched = ratings[ratings['userId'] == user_id]['movieId'].values
    # movies user haven't watched yet
    not_watched = [m for m in all_movie_ids if m not in watched]
    # predict rating for each movie
    predictions_list = []
    for movie_id in not_watched:
        pred = model.predict(user_id, movie_id)
        predictions_list.append((movie_id, pred.est))
    # sort from highest to lowest
    predictions_list.sort(key=lambda x: x[1], reverse=True)
    # get movie titles
    top_movies = predictions_list[:n]
    result = []
    for movie_id, score in top_movies:
       movie_info = movies[movies['movieId'] == movie_id]

       title = movie_info['title'].values[0]
       genres = movie_info['genres'].values[0]

       result.append({
        'title': title,
        'genres': genres,
        'predicted_rating':f"{score:.2f}" })
    return pd.DataFrame(result)

In [19]:
# show top 10 movies for user 1
recommendations=recommend_movies(user_id=1,n=10)
print("top Movie Recommendations :")
print(recommendations)

top Movie Recommendations :
                                               title  \
0                   Shawshank Redemption, The (1994)   
1                     Philadelphia Story, The (1940)   
2                                 Rear Window (1954)   
3                          North by Northwest (1959)   
4                                  Casablanca (1942)   
5                                      Brazil (1985)   
6                                     Amadeus (1984)   
7        Seven Samurai (Shichinin no samurai) (1954)   
8  Dr. Strangelove or: How I Learned to Stop Worr...   
9  Spirited Away (Sen to Chihiro no kamikakushi) ...   

                                      genres predicted_rating  
0                                crime drama             5.00  
1                       comedy drama romance             5.00  
2                           mystery thriller             5.00  
3  action adventure mystery romance thriller             5.00  
4                              dram

# Content-Based Filtering

In [20]:
# Check Dataset
print(data.shape)
data[['movieId', 'title', 'genres']].head()

(67898, 7)


,movieId,title,genres
0,1,Toy Story (1995),adventure animation children comedy fantasy
1,3,Grumpier Old Men (1995),comedy romance
2,6,Heat (1995),action crime thriller
3,47,Seven (a.k.a. Se7en) (1995),mystery thriller
4,50,"Usual Suspects, The (1995)",crime mystery thriller


In [21]:
# Remove Duplicate Movies
movies_content = data[['movieId', 'title', 'genres']].drop_duplicates()
movies_content.reset_index(drop=True, inplace=True)
movies_content.head()

,movieId,title,genres
0,1,Toy Story (1995),adventure animation children comedy fantasy
1,3,Grumpier Old Men (1995),comedy romance
2,6,Heat (1995),action crime thriller
3,47,Seven (a.k.a. Se7en) (1995),mystery thriller
4,50,"Usual Suspects, The (1995)",crime mystery thriller


In [22]:
# Handle Missing Values
movies_content['genres'] = movies_content['genres'].fillna('')

In [23]:
# Convert Genres to TF-IDF Matrix
tfidf = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf.fit_transform(movies_content['genres'])
print(genre_matrix.shape)

(1297, 21)


In [24]:
#Calculate Cosine Similarity
cosine_sim = cosine_similarity(genre_matrix, genre_matrix)
print(cosine_sim.shape)

(1297, 1297)


In [25]:
# Create Movie Index Mapping
indices = pd.Series(movies_content.index,index=movies_content['title']).drop_duplicates()
indices.head()

title
Toy Story (1995)               0
Grumpier Old Men (1995)        1
Heat (1995)                    2
Seven (a.k.a. Se7en) (1995)    3
Usual Suspects, The (1995)     4
dtype: int64

In [26]:
# Recommendation Function
def recommend_movies(title, cosine_sim=cosine_sim):
    # Check if movie exists
    if title not in indices:
        return "Movie not found"
    # Get movie index
    idx = indices[title]
    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Sort movies
    sim_scores = sorted(sim_scores,key=lambda x: x[1],reverse=True)
    # Remove selected movie
    sim_scores = sim_scores[1:11]
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    # Return recommendations
    return movies_content[['title', 'genres']].iloc[movie_indices]

In [27]:
# Test Recommendation System
recommend_movies('Toy Story (1995)')

,title,genres
435,Toy Story 2 (1999),adventure animation children comedy fantasy
450,"Monsters, Inc. (2001)",adventure animation children comedy fantasy
900,Antz (1998),adventure animation children comedy fantasy
967,"Emperor's New Groove, The (2000)",adventure animation children comedy fantasy
1022,Shrek the Third (2007),adventure animation children comedy fantasy
651,Inside Out (2015),adventure animation children comedy drama fantasy
1045,The Lego Movie (2014),action adventure animation children comedy fan...
444,Shrek (2001),adventure animation children comedy fantasy ro...
454,Ice Age (2002),adventure animation children comedy
521,Finding Nemo (2003),adventure animation children comedy


In [28]:
# Test Another Movie
recommend_movies('Jumanji (1995)')

,title,genres
308,Harry Potter and the Sorcerer's Stone (a.k.a. ...,adventure children fantasy
328,Jumanji (1995),adventure children fantasy
339,"Indian in the Cupboard, The (1995)",adventure children fantasy
1016,"Chronicles of Narnia: The Lion, the Witch and ...",adventure children fantasy
1233,Lemony Snicket's A Series of Unfortunate Event...,adventure children comedy fantasy
98,"Goonies, The (1985)",action adventure children comedy fantasy
210,Field of Dreams (1989),children drama fantasy
321,Addams Family Values (1993),children comedy fantasy
386,"Flintstones, The (1994)",children comedy fantasy
418,Matilda (1996),children comedy fantasy


In [29]:
#Recommendation Function with Similarity Score
def recommend_with_score(title, cosine_sim=cosine_sim):

    if title not in indices:
        return "Movie not found"

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    scores = [i[1] for i in sim_scores]

    result = movies_content[['title', 'genres']].iloc[movie_indices]

    result['similarity_score'] = scores

    return result

In [30]:
# Test Function with Scores
recommend_with_score('Heat (1995)')

,title,genres,similarity_score
31,Batman (1989),action crime thriller,1.0
183,Kill Bill: Vol. 1 (2003),action crime thriller,1.0
352,Die Hard: With a Vengeance (1995),action crime thriller,1.0
356,"Net, The (1995)",action crime thriller,1.0
374,Natural Born Killers (1994),action crime thriller,1.0
479,"Bourne Supremacy, The (2004)",action crime thriller,1.0
508,xXx (2002),action crime thriller,1.0
536,"Bourne Ultimatum, The (2007)",action crime thriller,1.0
610,Ronin (1998),action crime thriller,1.0
685,Die Hard (1988),action crime thriller,1.0


## Hybrid Recommendation (CF + CB)

In [31]:
def recommend_hybrid(user_id, title, n=10, cf_weight=0.6, cb_weight=0.4):
    all_movies  = ratings['movieId'].unique()
    watched     = ratings[ratings['userId'] == user_id]['movieId'].values
    not_watched = [m for m in all_movies if m not in watched]

    # CF scores (normalize 0 to 1)
    cf_scores = {m: model.predict(user_id, m).est for m in not_watched}
    cf_min, cf_max = min(cf_scores.values()), max(cf_scores.values())
    cf_norm = {m: (s - cf_min) / (cf_max - cf_min + 1e-9) for m, s in cf_scores.items()}

    # CB scores (normalize 0 to 1)
    cb_scores = {}
    if title in indices:
        idx = indices[title]
        for i, score in enumerate(cosine_sim[idx]):
            movie_id = movies_content.iloc[i]['movieId']
            if movie_id in cf_norm:
                cb_scores[movie_id] = score

    cb_vals = list(cb_scores.values()) if cb_scores else [0]
    cb_min, cb_max = min(cb_vals), max(cb_vals)
    cb_norm = {m: (s - cb_min) / (cb_max - cb_min + 1e-9) for m, s in cb_scores.items()}

    # combine scores
    hybrid_scores = {
        m: cf_weight * cf_norm[m] + cb_weight * cb_norm.get(m, 0)
        for m in cf_norm
    }

    top = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:n]
    result = []
    for movie_id, score in top:
        info = movies[movies['movieId'] == movie_id]
        if len(info) == 0:
            continue
        result.append({
            'title':         info['title'].values[0],
            'genres':        info['genres'].values[0],
            'hybrid_score':  round(score, 4)
        })

    return pd.DataFrame(result)

# test
print(recommend_hybrid(user_id=1, title='Toy Story (1995)', n=10))


                                               title  \
0                              Monsters, Inc. (2001)   
1  Spirited Away (Sen to Chihiro no kamikakushi) ...   
2  Laputa: Castle in the Sky (Tenkû no shiro Rapy...   
3                                 Toy Story 3 (2010)   
4                                 Toy Story 2 (1999)   
5                                          Up (2009)   
6        Wallace & Gromit: The Wrong Trousers (1993)   
7                   Emperor's New Groove, The (2000)   
8           Princess Mononoke (Mononoke-hime) (1997)   
9                    How to Train Your Dragon (2010)   

                                              genres  hybrid_score  
0        adventure animation children comedy fantasy        0.9276  
1                        adventure animation fantasy        0.9236  
2  action adventure animation children fantasy sc...        0.9155  
3   adventure animation children comedy fantasy imax        0.8979  
4        adventure animation children 

## Evaluation (Precision, Recall, F1)


In [32]:
def precision_recall_f1(predictions, k=10, threshold=3.5):
    df = pd.DataFrame(predictions, columns=['uid', 'iid', 'r_ui', 'est', 'details'])

    precisions, recalls = [], []

    for uid in df['uid'].unique():
        user_df = df[df['uid'] == uid].sort_values('est', ascending=False)
        top_k   = user_df.head(k)

        n_rel = (user_df['r_ui'] >= threshold).sum()
        n_hit = (top_k['r_ui']   >= threshold).sum()

        precisions.append(n_hit / k)
        recalls.append(n_hit / n_rel if n_rel > 0 else 0)

    p  = np.mean(precisions)
    r  = np.mean(recalls)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

    return p, r, f1

In [33]:
p, r, f1 = precision_recall_f1(predictions)
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Precision (Top 10): {p:.4f}")
print(f"Recall(Top 10) : {r:.4f}")
print(f"F1-Score(Top 10): {f1:.4f}")

RMSE: 0.8395
MAE: 0.6423
Precision (Top 10): 0.6005
Recall(Top 10) : 0.7013
F1-Score(Top 10): 0.6470
